In [ ]:
#showw people and id by a df
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('SparkSQL').getOrCreate()
people = spark.read.option('header', 'true').option('inferSchema', 'true').csv('fakefriends.csv')
print('Here is our inferred schema')
people.printSchema()

people.select('name','age').show()
people.groupBy('age').count().orderBy('age').show()
people.groupBy('age').agg(func.round(func.avg('friends'), 2)).orderBy('age').show()




In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('CustomerTotals').getOrCreate()
df = spark.read.csv(\
    'customer-orders.csv', 
    schema = "customerId STRING, orderId STRING, orderPrice DOUBLE"
    )
df.createOrReplaceTempView('orders')

customerTotal = spark.sql("""
    SELECT customerId, SUM(orderPrice) AS total
    FROM orders
    GROUP BY customerId
    ORDER BY total DESC
""")
customerTotal.show()
spark.stop()

+----------+------------------+
|customerId|             total|
+----------+------------------+
|        68| 6375.449999999997|
|        73| 6206.199999999999|
|        39| 6193.109999999999|
|        54| 6065.389999999999|
|        71| 5995.660000000003|
|         2|           5994.59|
|        97| 5977.189999999995|
|        46| 5963.109999999999|
|        42| 5696.840000000003|
|        59|           5642.89|
|        41|           5637.62|
|         0| 5524.949999999998|
|         8| 5517.240000000001|
|        85|           5503.43|
|        61| 5497.479999999998|
|        32| 5496.050000000004|
|        58|5437.7300000000005|
|        63| 5415.150000000001|
|        15| 5413.510000000001|
|         6| 5397.879999999998|
+----------+------------------+
only showing top 20 rows


In [ ]:
from pyspark.sql import Row, SparkSession
from pyspark.sql import functions as func
spark = SparkSession.builder.appName('ageFriendCount').getOrCreate()

def parseLine(line):
    fields = line.split(',')
    return Row(ID=int(fields[0]),name=str(fields[1].encode('utf-8')),age=int(fields[2]),friends=int(fields[3]))
lines = spark.sparkContext.textFile('fakefriends.csv')
people = lines.map(parseLine)
schemaPeople = spark.createDataFrame(people).cache()
schemaPeople.createOrReplaceTempView('people')
adults = spark.sql("""SELECT * FROM people WHERE age>=18""")
for p in adults.collect():
    print(p)

schemaPeople.groupBy('age').count().orderBy('age').show()
spark.stop()

In [6]:
from pyspark.sql import SparkSession,Row

spark = SparkSession.builder.appName('CustomRow').getOrCreate()
#memorize 2 lines above, smae with read operations
#if we dont infer schema, it defaults to string
people = spark.read.option('header', 'true').option('inferSchema', 'true').csv('fakefriends-header.csv')
people.show()
people.printSchema()

people.select('name').show()

people.filter(people.age > 35).show()
people.groupBy('age').count().show()

spark.stop()

+------+--------+---+-------+
|userID|    name|age|friends|
+------+--------+---+-------+
|     0|    Will| 33|    385|
|     1|Jean-Luc| 26|      2|
|     2|    Hugh| 55|    221|
|     3|  Deanna| 40|    465|
|     4|   Quark| 68|     21|
|     5|  Weyoun| 59|    318|
|     6|  Gowron| 37|    220|
|     7|    Will| 54|    307|
|     8|  Jadzia| 38|    380|
|     9|    Hugh| 27|    181|
|    10|     Odo| 53|    191|
|    11|     Ben| 57|    372|
|    12|   Keiko| 54|    253|
|    13|Jean-Luc| 56|    444|
|    14|    Hugh| 43|     49|
|    15|     Rom| 36|     49|
|    16|  Weyoun| 22|    323|
|    17|     Odo| 35|     13|
|    18|Jean-Luc| 45|    455|
|    19|  Geordi| 60|    246|
+------+--------+---+-------+
only showing top 20 rows
root
 |-- userID: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- friends: integer (nullable = true)

+--------+
|    name|
+--------+
|    Will|
|Jean-Luc|
|    Hugh|
|  Deanna|
|   Quark|
|  Weyoun|

#Lets look at some user defined functions

In [2]:
from pyspark.sql import SparkSession, Row
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
spark = SparkSession.builder.appName('UDF_example').getOrCreate()

@udf(returnType=StringType())
def user_details(name,age):
    return f"Name: {name}, Age: {age}"

people = (
    spark.read.option('header', 'true').option('inferSchema', 'true').csv('fakefriends-header.csv')
    #header is present, inferSchema is true, so it will automatically determine the data types of the columns
)
people.withColumn('details', user_details(people.name, people.age)).show()
spark.stop()


+------+--------+---+-------+--------------------+
|userID|    name|age|friends|             details|
+------+--------+---+-------+--------------------+
|     0|    Will| 33|    385| Name: Will, Age: 33|
|     1|Jean-Luc| 26|      2|Name: Jean-Luc, A...|
|     2|    Hugh| 55|    221| Name: Hugh, Age: 55|
|     3|  Deanna| 40|    465|Name: Deanna, Age...|
|     4|   Quark| 68|     21|Name: Quark, Age: 68|
|     5|  Weyoun| 59|    318|Name: Weyoun, Age...|
|     6|  Gowron| 37|    220|Name: Gowron, Age...|
|     7|    Will| 54|    307| Name: Will, Age: 54|
|     8|  Jadzia| 38|    380|Name: Jadzia, Age...|
|     9|    Hugh| 27|    181| Name: Hugh, Age: 27|
|    10|     Odo| 53|    191|  Name: Odo, Age: 53|
|    11|     Ben| 57|    372|  Name: Ben, Age: 57|
|    12|   Keiko| 54|    253|Name: Keiko, Age: 54|
|    13|Jean-Luc| 56|    444|Name: Jean-Luc, A...|
|    14|    Hugh| 43|     49| Name: Hugh, Age: 43|
|    15|     Rom| 36|     49|  Name: Rom, Age: 36|
|    16|  Weyoun| 22|    323|Na

In [8]:
from pyspark.sql import SparkSession, Row, functions as func

spark = SparkSession.builder.appName('WordCount').getOrCreate()
inputDF = spark.read.text('pride_and_prejudice.txt')
words = inputDF.select(func.explode(func.split(inputDF.value, r'\W+')).alias('word'))
wordsWithoutEmptyStrings = words.filter(words.word != "")
lowercaseWords = wordsWithoutEmptyStrings.select(func.lower(wordsWithoutEmptyStrings.word).alias('word'))
lowercaseWords.groupBy('word').count().orderBy(func.desc('count')).show()




spark.stop()

+----+-----+
|word|count|
+----+-----+
| the| 4846|
|  to| 4405|
|  of| 3962|
| and| 3835|
| her| 2260|
|   i| 2098|
|   a| 2094|
|  in| 2051|
| was| 1874|
| she| 1732|
|that| 1620|
|  it| 1603|
| not| 1520|
| you| 1417|
|  he| 1350|
| his| 1289|
|  be| 1280|
|  as| 1239|
| had| 1181|
|with| 1149|
+----+-----+
only showing top 20 rows


In [10]:
from pyspark.sql import SparkSession, functions as func
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType

spark = SparkSession.builder.appName('MinTemps').getOrCreate()

schema = StructType([
    StructField('StationID', StringType(), True),
    StructField('Date', IntegerType(), True),
    StructField('MeasureType', StringType(), True),
    StructField('Temperature', FloatType(), True)
])

df = spark.read.csv(
    'weatherData1800s.csv',
    schema=schema,
    header=False
)

minTemps = df.filter(df.MeasureType == 'TMIN')
minTempsByStation = minTemps.groupBy('StationID').min('Temperature')
minTempsByStation.orderBy('min(Temperature)').show()

+-----------+----------------+
|  StationID|min(Temperature)|
+-----------+----------------+
|ITE00100554|          -148.0|
|EZE00100082|          -135.0|
+-----------+----------------+



In [ ]:
from pyspark.sql import SparkSession, functions as func
from pyspark.sql.types import StructType, StructField, IntegerType, FloatType
spark = SparkSession.builder.appName('CustomerTotals').getOrCreate()
schema = StructType([
    StructField('customerID', IntegerType(), True),
    StructField('OrderId', IntegerType(), True),
    StructField('Amount', FloatType(), True)
    ])

df = spark.read.csv('customer-orders.csv', schema=schema)
df.createOrReplaceTempView('orders')
customerTotal = df.groupBy('customerID').agg(func.round(func.sum('Amount'), 2).alias('total')).orderBy(func.desc('total'))
customerTotal.show()
spark.stop()


+----------+------------------+
|customerID|             total|
+----------+------------------+
|        68| 6375.450028181076|
|        73| 6206.199985742569|
|        39| 6193.109993815422|
|        54| 6065.390002984554|
|        71| 5995.659991919994|
|         2| 5994.589979887009|
|        97| 5977.190007060766|
|        46| 5963.110011339188|
|        42| 5696.840004444122|
|        59| 5642.890004396439|
|        41| 5637.619991332293|
|         0| 5524.950008839369|
|         8|5517.2399980425835|
|        85|  5503.42998456955|
|        61| 5497.479998707771|
|        32| 5496.049998283386|
|        58| 5437.730004191399|
|        63| 5415.150004655123|
|        15| 5413.510010659695|
|         6| 5397.880012750626|
+----------+------------------+
only showing top 20 rows
